# Day 7 — LlamaIndex Fundamentals
---
> **Phase:** 1 — LLM Foundations  
> **Status:** ✅ Complete

## 🧠 Key Concepts

### 1. Why LlamaIndex Exists — The Problem It Solves

**The situation:** A company has 500 internal PDF documents. Employees need to ask questions and get answers from them.

**Why you can't just pipe documents into an LLM:**

Problem 1 — Token limits. 500 PDFs far exceeds any context window.

Problem 2 — Attention dilution. LLMs are built on Transformer self-attention — every token attends to every other token. Dumping 500 pages means irrelevant tokens compete for attention with relevant ones. The main context required for the answer gets diluted and the model produces low quality answers.

**The solution:** Find only the relevant pieces first, then give only those to the LLM.

**The tool for finding relevant pieces:** Semantic search — which you already built from scratch on Days 4 and 5.

```
The full pipeline you reasoned out yourself:
1. Split documents into chunks
2. Embed each chunk as a vector
3. Store vectors + text + metadata
4. User asks question → embed question → find similar chunks
5. Pass only relevant chunks to LLM
6. LLM synthesizes final answer

This is RAG — Retrieval Augmented Generation.
LlamaIndex manages this entire pipeline.
```

### 2. LangChain vs LlamaIndex — Different Problems

```
LangChain   → general orchestration framework
              connect LLMs, tools, memory, APIs
              you wire the pipeline yourself
              good for: agents, custom chains, full control

LlamaIndex  → data framework for LLMs
              connect your documents to LLMs
              pipeline is already built, deeply optimized
              good for: document ingestion, chunking, retrieval
```

**Production reality:** Many systems use both.
LlamaIndex for data ingestion and retrieval. LangChain for custom orchestration on top.

**The right mental model:**
- Need full control over the prompt? → LangChain
- Need intelligent document handling? → LlamaIndex
- Need both? → Use both together

### 3. The Chunking Problem

Before you can embed a document, you must split it into chunks. This is harder than it sounds.

**Why naive fixed-size chunking breaks:**

```
Problem 1 — Broken sentences
Chunk 1: "Employees are entitled to 18 days of paid leave per year, but this does not include"
Chunk 2: "public holidays which are separate and listed in Appendix B"
→ User asks "how many total days off?" → retrieves only chunk 1 → gets incomplete answer: 18 days

Problem 2 — Broken tables
Fixed character splitting tears tables apart
Headers get separated from data
The vector embedding of half a table means nothing semantically

Problem 3 — Embedding model hard limit
all-MiniLM-L6-v2 max input = 256 tokens
Pass a 50 page PDF as one string → model crashes or silently truncates
Chunking is not optional — it's required by the embedding model itself

Problem 4 — Reference problem
Legal contracts: "as defined in Section 4.2.1, the party shall..."
Fixed chunking splits the reference from its target
Retrieval finds the reference but Section 4.2.1 is in a low-scoring chunk
Answer is incomplete or wrong
```

**chunk_overlap solves Problem 1:**
```
chunk_size=256, chunk_overlap=20

Chunk 1: [tokens 1-256]
Chunk 2: [tokens 237-492]  ← starts 20 tokens before chunk 1 ended

The last 20 tokens of chunk 1 are repeated at the start of chunk 2
→ broken sentences get context from previous chunk
→ retrieval quality improves significantly
```

### 4. Documents and Nodes

**Document** = what you load. Raw content + metadata container.
- Not just a plain string
- Allows attaching metadata: filename, page number, author, date
- LlamaIndex needs Document objects, not raw strings

**Node** = what LlamaIndex creates internally from Documents.
```
Node contains:
  - text          → the chunk content
  - vector        → embedding of that chunk
  - metadata      → filename, page number, position
  - relationships → prev chunk, next chunk, parent document
```

**Why relationships matter — context window expansion:**
When retrieval finds a relevant chunk, LlamaIndex can automatically pull surrounding chunks using the prev/next relationships. Even if an answer was split across a chunk boundary, the full context can be recovered.

FAISS only stored vectors. You had to maintain a separate Python list and hope indices stayed aligned. Nodes store everything together — no alignment problem.

### 5. Index Types

An index is a data structure that organizes your chunks for a specific type of retrieval. Same documents, different organization, different retrieval strategy.

```
VectorStoreIndex   → semantic search using vectors
                     embed query → find similar vectors → return top K
                     use for: most production RAG systems

SummaryIndex       → no search, LLM reads ALL nodes sequentially
                     use for: summarization over small fixed document sets
                     expensive but thorough

TreeIndex          → builds summary tree bottom-up
                     leaf nodes → LLM summarizes groups → parent nodes
                     use for: very long hierarchical documents

KeywordTableIndex  → keyword matching, no vectors
                     use for: when exact keyword lookup is needed
```

**For most production work: VectorStoreIndex.**

### 6. Settings — Global Configuration

LlamaIndex does multiple things automatically — chunking, embedding, retrieval, synthesis. Several steps need to call a model internally without you explicitly telling it each time.

Settings is global configuration. Set it once, everything uses it automatically.

```python
Settings.llm          → which LLM to use for synthesis
Settings.embed_model  → which embedding model to use for vectorizing chunks
Settings.chunk_size   → how many tokens per chunk (match your embedding model limit)
Settings.chunk_overlap → how many tokens to repeat between chunks
```

Without Settings, LlamaIndex would use a random default LLM and fail on API keys or use the wrong embedding dimensions.

### 7. Query Engine — LlamaIndex's Built-in Search

On Day 5 you wrote a `search()` function yourself after building the FAISS index. `query_engine` is the same idea — LlamaIndex's built-in search function, already written for you.

```python
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query("How long does the battery last?")
```

**What happens inside `query_engine.query()`:**
```
1. Embed the query
2. Find top K similar nodes (similarity_top_k=3)
3. Build prompt internally:
   "Context: [chunk1] [chunk2] [chunk3]
    Question: How long does the battery last?"
4. Send prompt to LLM
5. Return Response object
```

**The Response object:**
```python
response.response      → synthesized answer from LLM (e.g. "All day.")
response.source_nodes  → list of Node objects that were retrieved
  node.text            → the raw chunk text
  node.score           → similarity score for this chunk
```

**Source nodes vs synthesized response:**
```
source_nodes     → what was RETRIEVED  (raw chunks, unprocessed)
response.response → what was SYNTHESIZED (LLM's answer based on those chunks)
```

The injection step (context + query → prompt) happens invisibly inside query_engine. On Day 6 with LangChain you built that manually with ChatPromptTemplate. Here LlamaIndex handles it automatically.

### 8. Persistence Pattern

Same logic as Day 5 FAISS pattern. Pay the embedding cost once, reload instantly every run after.

```python
# Save (after building)
index.storage_context.persist(persist_dir=storage_path)

# Load (every run after)
storage_context = StorageContext.from_defaults(persist_dir=storage_path)
index = load_index_from_storage(storage_context)
```

LlamaIndex saves more than FAISS did. Not just vectors — also node text, metadata, and relationships. The full structure, not just the binary vector file.

**Key difference from FAISS:**
```
FAISS           → saves to a single .index binary file
LlamaIndex      → saves to a directory (multiple files)
faiss.write_index(index, path)           → FAISS save
index.storage_context.persist(persist_dir=path) → LlamaIndex save
```

### 9. Query Wording Affects Retrieval

The query itself gets embedded into a vector. Different wording → different vector → different similarity scores.

```
"How long does the battery last?"    → battery review score: 0.5065
"how long will the battery last?"    → battery review score: 0.4941
```

Same document, same index, slightly different query phrasing, slightly different score.

**This is a real production problem.** Users ask the same question dozens of different ways. Some phrasings retrieve the right chunks, some don't.

**Solution coming in Phase 2:** Query rewriting — before searching, the LLM rewrites the user's question into the phrasing most likely to match indexed content.

## 💻 Code

In [ ]:
# Day 7 — LlamaIndex Persistent Query System
# review_query.py

from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, Document, StorageContext, load_index_from_storage, Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import os

load_dotenv()

# Global configuration — LlamaIndex uses these automatically
Settings.llm = Groq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"))
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
Settings.chunk_size = 256       # matches embedding model max input
Settings.chunk_overlap = 20     # last 20 tokens repeated in next chunk

texts = [
    "The battery life on this laptop is incredible, lasts all day",
    "Screen resolution is disappointing, colors look washed out",
    "Keyboard feels premium and typing experience is excellent",
    "The laptop runs very hot during heavy tasks, fan is loud",
    "Lightweight and portable, perfect for travel and commuting",
    "RAM is soldered and cannot be upgraded, big disappointment",
    "Boot time is blazing fast, SSD performance is outstanding",
    "Webcam quality is terrible, grainy even in good lighting",
    "Build quality feels solid, no flex in the chassis at all",
    "Price is too high for the specs you get with this machine"
]

script_dir = os.path.dirname(os.path.abspath(__file__))
index_path = os.path.join(script_dir, "reviews.index")

# Persistence pattern — same logic as Day 5 FAISS, different API
if os.path.exists(index_path):
    print("Loading existing index from disk.")
    storage_context = StorageContext.from_defaults(persist_dir=index_path)
    index = load_index_from_storage(storage_context)
    print("Index loaded.")
else:
    print("No index found. Building from scratch...")
    print("Embedding documents....")
    documents = [Document(text=t) for t in texts]  # plain strings → Document objects
    index = VectorStoreIndex.from_documents(documents)  # chunk → embed → store
    index.storage_context.persist(persist_dir=index_path)  # save everything to disk
    print("Index built and saved.")

# query_engine = LlamaIndex's built-in search function
# Same role as the search() function you wrote manually on Day 5
query_engine = index.as_query_engine(similarity_top_k=3)

print("\nSearch ready. Type 'quit' to exit.\n")

while True:
    query = input("Enter your query: ")
    print()

    if query.strip().lower() == "quit":
        break

    response = query_engine.query(query)

    # Synthesized answer — LLM read the chunks and produced this
    print(f"Answer: {response.response}")
    print()

    # Source nodes — the raw chunks that were retrieved
    print("Source chunks used:")
    for node in response.source_nodes:
        print(f"  Score: {node.score:.4f} | {node.text}")
    print()

## 🔬 Observations

| Run | What happened |
|-----|---------------|
| First run | Built index from scratch, embedded documents |
| Second run | Loaded from disk instantly, skipped embedding |
| Same query different wording | Same top result, slightly different score |

**Query wording observation:**
```
"How long does the battery last?"  → battery review score: 0.5065
"how long will the battery last?"  → battery review score: 0.4941
```
Different wording → different vector → different similarity score. Query phrasing matters in semantic search.

## ✅ Day 7 Summary

```
✓ Why RAG exists — attention dilution + token limits
✓ The chunking problem — why naive splitting breaks retrieval
✓ chunk_overlap — solves broken sentence problem
✓ Embedding model hard limit — chunking is required, not optional
✓ Reference problem — fixed chunking breaks cross-referencing documents
✓ Documents vs Nodes — text + vector + metadata + relationships
✓ Why Node relationships matter — context window expansion
✓ Four index types — VectorStore, Summary, Tree, Keyword
✓ Settings — global config so LlamaIndex knows which models to use
✓ query_engine — LlamaIndex's built-in search (same role as your Day 5 search())
✓ source_nodes vs response.response — retrieved vs synthesized
✓ Persistence pattern — same logic as Day 5, different API
✓ Query wording affects retrieval scores
✓ When to use LangChain vs LlamaIndex vs both
```

## 🔜 Coming in Day 8 — RAG from Scratch

You now have all pieces separately:
- Embeddings (Day 4)
- FAISS vector storage (Day 5)
- LangChain prompt control (Day 6)
- LlamaIndex document pipeline (Day 7)

Day 8 combines them. You'll build a full RAG system on real documents — and understand where naive RAG breaks and how to fix it.

**Pre-Day 8 Question:**

What kind of document would completely break a fixed chunk_size=256 approach?

Answer: Documents where structure and cross-references matter — legal contracts, academic papers, code documentation, financial reports. Fixed size chunking knows nothing about logical structure. It splits references from their targets. The chunk containing "as defined in Section 4.2.1" gets retrieved but Section 4.2.1 itself is in a low-scoring chunk that never gets retrieved. The answer is incomplete or wrong.

**Install command for Day 7:**
```bash
uv pip install llama-index llama-index-llms-groq llama-index-embeddings-huggingface
```